In [ ]:
%pip install pandas google-generativeai


In [ ]:
dbutils.library.restartPython()


In [ ]:
"""
InsightForge AI — Data Dictionary
===================================
Automatically generates a business description for every
column in the dataset using Gemini AI.

For each column it computes:
- Data type
- Unique value count
- Missing value count and percentage
- Sample values
- AI-generated business description

Author   : [Your Name]
Platform : Databricks
Model    : gemini-flash-latest
"""

import pandas as pd
import google.generativeai as genai
import time

# ── Widgets ───────────────────────────────────────────────────
dbutils.widgets.text(
    "dataset_path",
    "/Volumes/insight/default/titanic/Titanic.csv",
    "Dataset Path"
)
dbutils.widgets.text("gemini_key", "", "Gemini API Key")

DATASET_PATH = dbutils.widgets.get("dataset_path")
GEMINI_KEY   = dbutils.widgets.get("gemini_key")
GEMINI_MODEL = "gemini-flash-latest"

# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)

# ── Configure Gemini ──────────────────────────────────────────
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)

print("=" * 55)
print("  InsightForge AI — Data Dictionary")
print("=" * 55)
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Shape    : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Columns  : {df.columns.tolist()}")
print(f"  Model    : {GEMINI_MODEL}")
print("=" * 55)


In [ ]:
def get_column_stats(df: pd.DataFrame, col: str) -> dict:
    """
    Computes statistical metadata for a single column.
    Called once per column before the Gemini API call.
    Separates data computation from AI description so
    each concern is handled independently.

    Parameters
    ----------
    df  : pd.DataFrame — the dataset
    col : str          — column name to analyse

    Returns
    -------
    dict : column metadata including type, nulls, samples
    """
    series      = df[col]
    total_rows  = len(series)
    null_count  = int(series.isnull().sum())
    null_pct    = round(null_count / total_rows * 100, 2)

    # Get clean sample values — drop nulls first
    sample_vals = series.dropna().unique()[:5].tolist()

    # Convert numpy types to native Python for clean display
    sample_vals = [
        float(v) if hasattr(v, "item") else v
        for v in sample_vals
    ]

    stats = {
        "column"      : col,
        "dtype"       : str(series.dtype),
        "total_rows"  : total_rows,
        "null_count"  : null_count,
        "null_pct"    : null_pct,
        "unique_count": int(series.nunique()),
        "sample_values": sample_vals,
    }

    # Add numeric-specific stats
    if series.dtype in ("int64", "float64"):
        stats["min"]  = round(float(series.min()), 4) if not series.isnull().all() else None
        stats["max"]  = round(float(series.max()), 4) if not series.isnull().all() else None
        stats["mean"] = round(float(series.mean()), 4) if not series.isnull().all() else None

    # Add categorical-specific stats
    else:
        top_values = series.value_counts().head(3).to_dict()
        stats["top_values"] = {str(k): int(v) for k, v in top_values.items()}

    return stats


# Test on one column
test_stats = get_column_stats(df, "Age")
print("✅ get_column_stats() defined")
print()
print("Test on 'Age' column:")
for k, v in test_stats.items():
    print(f"  {k:15} : {v}")


In [ ]:
def generate_column_description(
    stats          : dict,
    dataset_context: str
) -> str:
    """
    Asks Gemini to explain what a column means in business terms.
    Includes automatic retry with wait on rate limit errors.

    Parameters
    ----------
    stats           : dict — output of get_column_stats()
    dataset_context : str  — brief description of the full dataset

    Returns
    -------
    str : one clear sentence explaining the column's business meaning
    """
    col    = stats["column"]
    dtype  = stats["dtype"]
    sample = stats["sample_values"]
    nulls  = stats["null_pct"]

    if "mean" in stats:
        extra = (
            f"Min: {stats['min']}, Max: {stats['max']}, "
            f"Mean: {stats['mean']}"
        )
    elif "top_values" in stats:
        extra = f"Most common values: {stats['top_values']}"
    else:
        extra = ""

    prompt = f"""
You are a data dictionary writer for a business audience.

Dataset context: {dataset_context}

Explain this column in ONE clear sentence for a business user
who is not a data scientist. Be specific about what the values mean.

Column details:
- Name          : {col}
- Data type     : {dtype}
- Sample values : {sample}
- Missing values: {nulls}%
- {extra}

Write ONE sentence only. Start with "{col} represents..." or
"{col} indicates..." or "{col} contains...".
Do not use technical jargon.
"""

    # ── Retry logic with exponential backoff ──────────────────
    max_retries = 3
    wait_seconds = 20  # start with 20 seconds on first retry

    for attempt in range(1, max_retries + 1):
        try:
            response = model.generate_content(prompt)
            time.sleep(4)  # 4 second gap between calls
                           # 15 calls/min limit → 4s gap = safe
            return response.text.strip()

        except Exception as e:
            error_str = str(e)

            # Check if it is a rate limit error
            if "429" in error_str or "quota" in error_str.lower():
                if attempt < max_retries:
                    print(
                        f"\n   ⚠️  Rate limit hit on '{col}' "
                        f"— waiting {wait_seconds}s "
                        f"(attempt {attempt}/{max_retries})..."
                    )
                    time.sleep(wait_seconds)
                    wait_seconds *= 2  # double wait on next retry
                else:
                    print(f"\n   ❌ Rate limit: max retries reached for '{col}'")
                    return f"Rate limit reached — retry later"
            else:
                # Non-rate-limit error — do not retry
                print(f"\n   ❌ API error for '{col}': {e}")
                return f"Description unavailable: {str(e)}"

    return "Description unavailable after retries"

print("✅ generate_column_description() updated with retry logic")
print("   Retry strategy: 3 attempts, 20s → 40s → 80s wait")
print("   Gap between calls: 4 seconds")


In [ ]:
def generate_data_dictionary(
    df             : pd.DataFrame,
    dataset_context: str = "A tabular dataset"
) -> pd.DataFrame:
    """
    Generates a complete data dictionary for every column.

    For each column:
    1. Computes structural stats (type, nulls, samples)
    2. Calls Gemini for a business-readable description
    3. Assembles everything into a clean DataFrame

    Parameters
    ----------
    df              : pd.DataFrame — the dataset to document
    dataset_context : str — one sentence describing the dataset

    Returns
    -------
    pd.DataFrame : one row per column with full metadata
    """
    total = df.shape[1]
    print(f"Generating data dictionary for {total} columns...")
    print(f"Dataset context : {dataset_context}")
    print(f"Rate limit note : 4 second gap between Gemini calls")
    print(f"Estimated time  : ~{total * 5} seconds")
    print("─" * 55)

    records = []

    for i, col in enumerate(df.columns, 1):
        print(f"  [{i:2}/{total}] '{col}'...", end=" ", flush=True)

        # Step 1 — compute stats (no API call)
        stats = get_column_stats(df, col)

        # Step 2 — get AI description (API call with retry)
        description = generate_column_description(
            stats           = stats,
            dataset_context = dataset_context
        )

        # Step 3 — build record
        record = {
            "Column"        : col,
            "Type"          : stats["dtype"],
            "Non-Null"      : stats["total_rows"] - stats["null_count"],
            "Missing"       : stats["null_count"],
            "Missing %"     : stats["null_pct"],
            "Unique Values" : stats["unique_count"],
            "Sample Values" : str(stats["sample_values"]),
            "Description"   : description
        }

        if "mean" in stats:
            record["Range"] = (
                f"{stats['min']} to {stats['max']}"
                if stats["min"] is not None else "N/A"
            )
        else:
            top = stats.get("top_values", {})
            record["Range"] = str(list(top.keys())[:3])

        records.append(record)
        print("✓")

    dd = pd.DataFrame(records)
    dd = dd.set_index("Column")

    print()
    print(f"✅ Data dictionary complete — {len(dd)} columns documented")
    return dd

print("✅ generate_data_dictionary() updated")


In [ ]:
# Run the data dictionary on Titanic
data_dict = generate_data_dictionary(
    df              = df,
    dataset_context = (
        "Passenger records from the RMS Titanic disaster of 1912, "
        "containing demographic and travel information with survival outcomes"
    )
)

print()
print("=" * 55)
print("DATA DICTIONARY")
print("=" * 55)
display(data_dict)


In [ ]:
# Print in a readable format — useful for reports
print("DATA DICTIONARY — READABLE FORMAT")
print("=" * 55)

for col_name, row in data_dict.iterrows():
    print(f"\n  {col_name}")
    print(f"    Type         : {row['Type']}")
    print(f"    Non-Null     : {row['Non-Null']} / {df.shape[0]}")
    print(f"    Missing      : {row['Missing']} ({row['Missing %']}%)")
    print(f"    Unique values: {row['Unique Values']}")
    print(f"    Range/Top    : {row['Range']}")
    print(f"    Sample       : {row['Sample Values']}")
    print(f"    Description  : {row['Description']}")
    print(f"    {'─' * 48}")


In [ ]:
# Save as CSV to Unity Catalog Volume
output_path = "/Volumes/insight/default/titanic/data_dictionary.csv"

data_dict.to_csv(output_path)

print(f"✅ Data dictionary saved to:")
print(f"   {output_path}")

# Verify it saved
files = dbutils.fs.ls("/Volumes/insight/default/titanic/")
for f in files:
    if "dictionary" in f.name.lower():
        size_kb = round(f.size / 1024, 2)
        print(f"   Confirmed: {f.name} — {size_kb} KB")


In [ ]:
def lookup_column(col_name: str) -> None:
    """
    Looks up a single column in the data dictionary and
    prints its full entry in a readable format.
    Useful for quick reference during analysis.

    Parameters
    ----------
    col_name : str — exact column name to look up
    """
    if col_name not in data_dict.index:
        print(f"❌ Column '{col_name}' not found in data dictionary")
        print(f"   Available columns: {data_dict.index.tolist()}")
        return

    row = data_dict.loc[col_name]
    print(f"\n  Column: {col_name}")
    print(f"  {'─' * 40}")
    for field, value in row.items():
        print(f"  {field:15} : {value}")


# Test lookups
lookup_column("Survived")
lookup_column("Fare")
lookup_column("Pclass")


In [ ]:
import pandas as pd
df = pd.read_csv("/Volumes/insight/default/titanic/Titanic.csv")
print(df.shape)
print(df.columns.tolist())
